# Milestone 2 — RAG Pipeline Exploration

This notebook walks through the full RAG pipeline:
1. Load Milestone 1 retrievers from disk.
2. Wrap them as LangChain `BaseRetriever` subclasses.
3. Build the context from retrieved documents.
4. Compare the three prompt variants on the same query.
5. Run end-to-end RAG over 10 evaluation queries.

**Prereqs:** `make setup` has been run, `HF_TOKEN` is set in `.env`, and the token has accepted the Meta Llama 3 license.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

In [ ]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

INDICES = Path.cwd().parent / "indices"

bm25 = BM25Retriever()
bm25.load(str(INDICES / "bm25_index.pkl"))

semantic = SemanticRetriever()
semantic.load(str(INDICES / "faiss_index"))

print(f"BM25 corpus: {len(bm25.corpus):,} products")
print(f"Semantic corpus: {len(semantic.corpus):,} products")

In [ ]:
from src.retrievers_lc import wrap_retriever

bm25_lc = wrap_retriever("BM25", bm25, semantic, top_k=5)
semantic_lc = wrap_retriever("Semantic", bm25, semantic, top_k=5)
hybrid_lc = wrap_retriever("Hybrid", bm25, semantic, top_k=5)

query = "vitamin c serum for dark spots"
for name, r in [("BM25", bm25_lc), ("Semantic", semantic_lc), ("Hybrid (RRF)", hybrid_lc)]:
    docs = r.invoke(query)
    print(f"
{name}:")
    for d in docs[:3]:
        print(f"  [{d.metadata["parent_asin"]}] {d.metadata["title"]}")

In [ ]:
from src.prompts import build_context

docs = hybrid_lc.invoke(query)
print(build_context(docs))

## Prompt Variant Comparison

Same query, three system prompts. We expect:
- `strict_citation`: short, ASIN-cited, may say "I don't have enough information."
- `helpful_shopper`: more conversational, may include price/rating commentary.
- `structured_json`: JSON object with `recommendation`, `reasoning`, `asins`.

In [ ]:
from src.rag_pipeline import RAGPipeline

comparison_query = "what's a good vitamin C serum for dark spots under $30?"

for variant in ["strict_citation", "helpful_shopper", "structured_json"]:
    pipeline = RAGPipeline(
        bm25=bm25, semantic=semantic,
        retriever_name="Hybrid", prompt_name=variant, top_k=5,
    )
    result = pipeline.answer(comparison_query)
    print(f"
=== {variant} ===")
    print(result["answer"])

## Evaluation Run (10 Queries)

Run the default pipeline (Hybrid + strict_citation + top_k=5) over the 10 RAG queries.
Outputs go to `data/eval_outputs/rag_eval.json` for hand-rating in `results/milestone2_discussion.md`.

In [ ]:
import json
import pandas as pd

EVAL_OUT = Path.cwd().parent / "data" / "eval_outputs" / "rag_eval.json"
EVAL_OUT.parent.mkdir(parents=True, exist_ok=True)

queries_df = pd.read_csv(Path.cwd().parent / "data" / "processed" / "rag_queries.csv")

default_pipeline = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation", top_k=5,
)

results = []
for _, row in queries_df.iterrows():
    print(f"Query {row["query_id"]}: {row["query"]}")
    answer = default_pipeline.answer(row["query"])
    results.append({
        "query_id": int(row["query_id"]),
        "query": row["query"],
        "category": row["category"],
        "answer": answer["answer"],
        "sources": answer["sources"],
    })

EVAL_OUT.write_text(json.dumps(results, indent=2))
print(f"
Saved {len(results)} eval outputs to {EVAL_OUT}")